# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/KhanhGiauTen/flyrankAI/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This notebook frames the project before any modeling. It uses only the public-safe starter dataset and treats every result as decision support, not causal proof.

## 1. My lane (or freestyle) and why

I choose the **Refresh / Content Opportunity Scoring** lane. The practical question is: **which pages should a content team inspect first when review capacity is limited?** This lane fits the available page-level search, engagement, age, and update signals, and it produces a ranked queue that a human can act on. The goal is not to automate editorial judgment; it is to spend scarce review time on the pages with the strongest measured evidence of a refresh opportunity.

In [1]:
from pathlib import Path
import pandas as pd

DATA_URL = 'https://raw.githubusercontent.com/KhanhGiauTen/flyrankAI/main/data/raw/content_refresh_anonymized.csv'
local_candidates = [Path('data/raw/content_refresh_anonymized.csv'), Path('../../data/raw/content_refresh_anonymized.csv')]
data_source = next((path for path in local_candidates if path.exists()), DATA_URL)
df = pd.read_csv(data_source)
print(f'Loaded {len(df):,} public-safe content rows across {df.client_id.nunique()} pseudonymized clients.')


Loaded 30,000 public-safe content rows across 32 pseudonymized clients.


## 2. The question: decision, action, cost of a wrong call

The decision is **which pages enter the next content-review batch, and in what order**. A content strategist or editor reviews the ranked list, checks page context, and chooses an action such as update, consolidate, improve the search snippet, or leave unchanged. A false positive wastes editorial time and may trigger an unnecessary change; a false negative leaves a real opportunity unattended. Because the team can review only a small batch, false positives near the top of the queue are especially costly, so the evaluation must focus on top-of-list precision rather than generic accuracy.

In [2]:
decision_capacity = 50
eligible = df[(df['impressions_90d'] >= 100) & (df['sessions_90d'] > 0)].copy()
print(f'Decision capacity: review {decision_capacity} pages per batch.')
print(f'Eligible measurable pages: {len(eligible):,}; capacity covers {decision_capacity / len(eligible):.2%} of them.')


Decision capacity: review 50 pages per batch.
Eligible measurable pages: 22,006; capacity covers 0.23% of them.


## 3. Quick look at the data (2-3 real numbers)

The starter snapshot contains **30,000 content items from 32 pseudonymized clients**. **16,262 pages (54.2%)** are labeled as declining by the teaching proxy, and **16,726 pages** have at least 500 impressions in the trailing 90 days. There are **22,006 measurable opportunities** with at least 100 impressions and at least one session. Those counts show both a large review backlog and enough visible pages to make prioritization useful.

In [3]:
evidence = pd.Series({
    'content_rows': len(df),
    'pseudonymized_clients': df['client_id'].nunique(),
    'declining_proxy_rows': int(df['trend_direction'].eq('down').sum()),
    'declining_proxy_rate_pct': round(df['trend_direction'].eq('down').mean() * 100, 1),
    'pages_with_500plus_impressions': int(df['impressions_90d'].ge(500).sum()),
    'measurable_opportunities': int(((df['impressions_90d'] >= 100) & (df['sessions_90d'] > 0)).sum()),
})
evidence.to_frame('observed_value')


,observed_value
content_rows,30000.0
pseudonymized_clients,32.0
declining_proxy_rows,16262.0
declining_proxy_rate_pct,54.2
pages_with_500plus_impressions,16726.0
measurable_opportunities,22006.0


## 4. Careful words: what I can and can't claim

This work can describe **observed associations** in the snapshot, compare a transparent rule with a model under client-grouped validation, and provide a **directional, decision-support ranking** for human review. It cannot prove that editing a page causes recovery, reveal Google's ranking factors, or guarantee that a page will decline. The starter `trend_direction == 'down'` label is a current-window teaching proxy, not a future outcome. A stronger capstone should separate historical feature windows from a later target window and preserve human review before any content action.

In [4]:
assert {'content_id', 'client_id', 'trend_direction', 'trend_pct'}.issubset(df.columns)
assert df['content_id'].is_unique, 'Expected one row per content item in the starter snapshot.'
assert not df['content_id'].astype(str).str.contains(r'https?://', regex=True).any()
print('Public-safety check passed: identifiers are pseudonymous and no URL appears in content_id.')
print('Claim boundary recorded: association and prioritization, not causation.')


Public-safety check passed: identifiers are pseudonymous and no URL appears in content_id.
Claim boundary recorded: association and prioritization, not causation.


## Self-check

- [x] Every section above is filled — markdown thinking and supporting code
- [x] The notebook runs top to bottom with no errors
- [x] No client names, URLs, or private queries appear in the output
- [x] Claims use careful words: observed, measured, directional, decision-support
- [x] Saved under `work/notebooks/` for the repository submission